# Signal

> Signal processing utilities that make complete sense

In [ ]:
#| default_exp signal

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from scipy import signal, interpolate
import numpy as np
from fractions import Fraction

In [ ]:
#| export
def butterworth(waveform_array, freq_range, btype, fs=128, order=4): # Recommend playing around with the order as well
    # Butterworth filter
    sos = signal.butter(order, freq_range, fs=fs, btype=btype, output='sos')
    filtered = signal.sosfiltfilt(sos, waveform_array) # zero phase filter (no phase shift)
    return filtered

def high_frequency_noise_filter(data):
    order, normal_cutoff = signal.buttord(20, 30, gpass=0.1, gstop=20, fs=240)
    iir_b, iir_a = signal.butter(order, normal_cutoff, fs=240)
    filtered_data = signal.filtfilt(iir_b, iir_a, data)
    return filtered_data

def baseline_filter(data):
    order, normal_cutoff = signal.buttord(0.5, 8, gpass=0.1, gstop=20, fs=240)
    iir_b, iir_a = signal.butter(order, normal_cutoff, fs=240)
    filtered_data = signal.filtfilt(iir_b, iir_a, data)
    return filtered_data

def resample_waveform(waveform_array, fs_in, fs_out, is_spo2=False):
    if not is_spo2:
        resample_fraction = Fraction(fs_out, fs_in)#.limit_denominator(100)
        resampled_waveform = signal.resample_poly(waveform_array, resample_fraction.numerator, resample_fraction.denominator)
    else:
        # linear interpolation
        t = np.arange(0, len(waveform_array)*(1/fs_in), 1/fs_in)
        resample_f = interpolate.make_interp_spline(t, waveform_array, k=1) # linear interpolation
        t_new = np.arange(0, len(waveform_array)*(1/fs_in), 1/fs_out)
        resampled_waveform = resample_f(t_new)
    return resampled_waveform


def iir_filter(waveform_array, freq_range, btype, order=16, fs=128):
    "as described in https://www.researchsquare.com/article/rs-6307069/v1"
    sos = signal.iirfilter(N=order, Wn=freq_range, rp=1, rs=40, btype=btype, analog=False, ftype='ellip', output='sos', fs=fs)
    filtered_data = signal.sosfiltfilt(sos, waveform_array)
    return filtered_data

def iqr_normalization(waveform_array, is_spo2=False):
    eps = 1e-10
    if not is_spo2:
        q5 = np.percentile(waveform_array, 5)
        q95 = np.percentile(waveform_array, 95)
        waveform_array = 2*(waveform_array - q5) / (q95 - q5 + eps) - 1 # scaled to -1 to 1
    else:
        # spo2 is scaled to 0.60-1.0
        if np.mean(waveform_array) > 1.0:
            waveform_array = waveform_array / 100.
        waveform_array = waveform_array.clip(0.6, 1.0)
        waveform_array = 2*(waveform_array - 0.6) / (1.0 - 0.6) - 1 # scaled to -1 to 1
    return waveform_array

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()